# Stateless Streaming Practice

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
schema=StructType([
    StructField("customer",StructType(
        [
            StructField("customer_id",IntegerType()),
            StructField("email",StringType()),
            StructField("name",StringType()),
            StructField("address",StructType([
                StructField("city",StringType()),
                StructField("country",StringType()),
                StructField("postal_code",StringType())
            ]))
        ]
    )
         
    ),
    StructField("items",ArrayType(
        StructType([
            StructField("item_id",StringType()),
            StructField("price",DoubleType()),
            StructField("product_name",StringType()),
            StructField("quantity",IntegerType())
            ])
    )),
    StructField("metadata",ArrayType(StructType([StructField("key",StringType()),
        StructField("value",StringType())
]))),
    StructField("order_id",StringType()),
    StructField("timestamp",StringType()),
    StructField("payment",StructType([
        StructField("method",StringType()),
        StructField("transaction_id",StringType())
    ]))
     
])

In [0]:
df1=spark.readStream.option('multiline',True).schema(schema).json("/Volumes/pyspark_catalog/moon/files/source/*.json") 
df1=df1.select("customer.customer_id","customer.email","customer.name","customer.address.city","customer.address.country","customer.address.postal_code","payment.method","payment.transaction_id","items","metadata","order_id","timestamp") 
df1=df1.withColumn("items",F.explode("items")).withColumn("metadata",F.explode_outer("metadata"))
df1=df1.select("customer_id","order_id","email","name","city","country","method","postal_code","transaction_id","items.item_id","items.price","items.product_name","items.quantity","metadata.key","metadata.value","timestamp")

In [0]:
df1.writeStream\
    .trigger(once=True)\
    .outputMode('append')\
    .option("path","/Volumes/pyspark_catalog/moon/files/target/")\
    .option("checkpointLocation","/Volumes/pyspark_catalog/moon/files/check/")\
    .start()

In [0]:
%python
df=spark.sql(f"select * from delta.`/Volumes/pyspark_catalog/moon/files/target/`")
df.show()

customer_id | order_id | email             | name        | city      | country | method     | postal_code | transaction_id | item_id | price  | product_name                | quantity | key      | value        | timestamp           
------------+----------+-------------------+-------------+-----------+---------+------------+-------------+----------------+---------+--------+-----------------------------+----------+----------+--------------+---------------------
503         | ORD1003  | david@example.com | David Lee   | Calgary   | Canada  | Debit Card | T2P 1G1     | TXN7892        | I103    | 199.99 | Noise Cancelling Headphones | 1        | referrer | instagram    | 2025-06-01T11:00:00Z
503         | ORD1003  | david@example.com | David Lee   | Calgary   | Canada  | Debit Card | T2P 1G1     | TXN7892        | I103    | 199.99 | Noise Cancelling Headphones | 1        | coupon   | WELCOME10    | 2025-06-01T11:00:00Z
503         | ORD1003  | david@example.com | David Lee   | Calgary   | C